In [1]:
import pandas as pd
import time
import random
import math
import mmh3
import statistics

In [2]:
def str_to_MinHash(str1, q, seed=0):
    return min([mmh3.hash(str1[i:i + q], seed) for i in range(len(str1) - q + 1)])

def frequent2(temp, L, t):
    return {k: v for (k, v) in temp.items() if v/L >= t}


In [25]:
def matching():
    global tp, fp, pairsNo, L1, q
    for index2 in range(nbS, nbS + offsetB):  # DBLP
        if index2 > len(df2) - 1:
            return True

        rr = df2.iloc[index2]
        ncid = rr["TID2"]
        title = rr["title"]
        length = rr["length"]
        artist = rr["artist"]
        album = rr["album"]
        srec = title + " " + length + " " + str(artist) + " " + str(album)

        temp = dict()
        indices = [random.randrange(0, L) for i in range(L1)]
        matchingPairs = {}
        for l in indices:
            key = str(str_to_MinHash(srec.lower(), q, l))
            d = dictB[l]
            if key in d:
                ids = d[key]
                for id in ids:
                    if id in temp:
                        temp[id] += 1
                        if temp[id] / L1 >= t:
                            matchingPairs[id] = 1
                    else:
                        temp[id] = 1
        for id in matchingPairs.keys():
            idDBLP = id
            pairsNo += 1
            if idDBLP in truthD:
                ids = truthD[idDBLP]
                for id in ids:
                    if id == ncid:
                        tp += 1
                        break
            else:
                fp += 1

    return False



In [26]:
df1 = pd.read_csv("../00-datasets/musicbrainz-200-A01.csv.dapo", sep=",", encoding="utf-8", keep_default_na=False)
df2 = pd.read_csv("../00-datasets/music_brainz-simple-mutated.csv", sep=",", encoding="utf-8", keep_default_na=False)

truth = pd.read_csv("../00-datasets/ground_truth_music_brainz.csv", sep=",", encoding="utf-8", keep_default_na=False)
truthD = dict()
for i, r in truth.iterrows():
    novoId = r["novoidmusic1"]
    antigoId = r["antigoidmusic2"]
    if antigoId in truthD:
        ids = truthD[antigoId]
        ids.append(novoId)
    else:
        truthD[antigoId] = [novoId]

df1 = df1.iloc[0:1000]
df2 = df2.iloc[0:1000]

t = 0.5
TP = 193471
eps = 0.1
w = 1000
delta = 0.1
L = math.ceil(math.log(1 / delta) / (2 * (eps ** 2)))
eps = 0.01
L1 = int(1 / (2 * eps))
print("L=", L, "L1=", L1)
q = 2

L= 116 L1= 50


In [28]:
dictB = [dict() for l in range(L)]
dictB_igual = [dict() for l in range(L)]

tp = 0
fp = 0
pairsNo = 0
nbS = 1
naS = 1
offsetA = 50
offsetB = 50
blockingTime = 0
matchingTime = 0

while True:
    st = time.time()
    for index1 in range(naS, naS + offsetA):  # Scholar
        if index1 >= len(df1):
            break
        rr = df1.iloc[index1]
        ncid = rr["TID"]
        title = rr["title"]
        length = rr["length"]
        artist = rr["artist"]
        album = rr["album"]
        srec = title + " " + length + " " + str(artist) + " " + str(album)
        key = ""
        
        for l in range(L):
            key = str(str_to_MinHash(srec.lower(), 2, l))
            d = dictB[l]
            
            if key in d:
                ids = d[key]

                ids.append(df1.iloc[index1, 0])
            else:
                d[key] = [df1.iloc[index1, 0]]
    end = time.time()
    
    blockingTime += (end - st)
    st = time.time()
    termination = matching()  
    end = time.time()
    matchingTime += (end - st)
    if termination:
            break

    nbS += offsetB
    naS += offsetA

print("blocking time (in mins)", blockingTime / 60)
print("matching time (in mins)", matchingTime / 60)
if tp + fp > 0:
    print("TP=", tp, "Recall=", tp / TP, "Precision=", tp / (tp + fp), "pairsNo=", pairsNo)


blocking time (in mins) 0.05980105400085449
matching time (in mins) 0.02332691748936971
TP= 756 Recall= 0.00390756237368908 Precision= 0.9908256880733946 pairsNo= 767
